[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-USERNAME/AI-in-healthcare-book/blob/main/notebooks/chapter_05/notebook_5_6_fairness_metrics.ipynb)

*Click the badge above to open this notebook in Google Colab (no setup required)*

---


# 5.6 Fairness Metrics and Bias Detection

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Measure stratified performance** across demographic groups
2. **Calculate demographic parity** and understand fairness definitions
3. **Calculate equalized odds** and equality of opportunity
4. **Identify algorithmic bias** in healthcare AI models
5. **Visualize fairness gaps** and disparities
6. **Understand trade-offs** between fairness and accuracy

---

## Clinical Context: Why Fairness Matters

**Journey Connection:**
- **Priya (Melanoma)**: AI performs worse on darker skin tones → dangerous for diverse populations

**Real-World Examples of Algorithmic Bias:**
- **Pulse oximeters**: Overestimate oxygen levels in Black patients → delayed treatment
- **Risk prediction**: Models underestimate severity for minority groups
- **Skin cancer detection**: Lower accuracy for darker skin tones (Fitzpatrick V-VI)
- **EHR algorithms**: Bias due to historical healthcare disparities in training data

**Key Questions:**
- Does the model perform equally well across all demographic groups?
- Are false positive/negative rates similar across groups?
- Is the model systematically disadvantaging any population?
- How do we balance fairness with overall accuracy?

**Ethical Stakes:**
- Biased AI can worsen existing health disparities
- Regulatory requirements (FDA guidance on fairness)
- Legal liability for discriminatory algorithms
- Trust and acceptance by diverse patient populations

---

## Setup

In [ ]:
# Standard imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, classification_report
)
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 10

# Set random seed
np.random.seed(42)

print("Setup complete!")

---

## Part 1: Stratified Performance Analysis

### The First Step: Measure Performance by Group

**Before worrying about formal fairness metrics, ask:**
1. Does the model perform equally well on different groups?
2. Are there large gaps in accuracy, sensitivity, or specificity?
3. Which groups are underserved?

**Protected attributes** (sensitive groups to analyze):
- Race/ethnicity
- Gender
- Age groups
- Socioeconomic status
- Geographic location
- Insurance type (proxy for SES)

### 1.1 Generate Synthetic Melanoma Detection Data with Bias

In [ ]:
def generate_melanoma_data_with_bias(n_samples=2000):
    """
    Generate synthetic melanoma detection data with skin tone bias.

    Simulates Priya's story: AI trained primarily on lighter skin tones
    performs worse on darker skin tones.

    Skin tone groups (Fitzpatrick scale):
    - Light: I, II, III (60% of dataset)
    - Dark: IV, V, VI (40% of dataset)

    Bias: Model has 90% sensitivity on light skin, 70% on dark skin
    """
    # Generate patient demographics
    skin_tone = np.random.choice(['Light', 'Dark'], size=n_samples, p=[0.6, 0.4])

    # Generate true melanoma status (10% prevalence, equal across groups)
    true_melanoma = np.random.random(n_samples) < 0.10

    # Generate features (lesion characteristics)
    # Feature engineering: darker skin tones have different presentation
    asymmetry = np.random.normal(0.5, 0.2, n_samples)
    border = np.random.normal(0.5, 0.2, n_samples)
    color = np.random.normal(0.5, 0.2, n_samples)
    diameter = np.random.normal(6, 2, n_samples)  # mm

    # Add signal for melanoma
    asymmetry[true_melanoma] += 0.3
    border[true_melanoma] += 0.3
    color[true_melanoma] += 0.3
    diameter[true_melanoma] += 2

    # Create biased predictions
    # Model is better at detecting melanoma on light skin
    predictions = np.zeros(n_samples, dtype=bool)

    for i in range(n_samples):
        if true_melanoma[i]:
            # Sensitivity varies by skin tone
            if skin_tone[i] == 'Light':
                predictions[i] = np.random.random() < 0.90  # 90% sensitivity
            else:
                predictions[i] = np.random.random() < 0.70  # 70% sensitivity (BIAS!)
        else:
            # Specificity is similar (85%)
            predictions[i] = np.random.random() > 0.85

    # Create DataFrame
    df = pd.DataFrame({
        'skin_tone': skin_tone,
        'asymmetry': asymmetry,
        'border': border,
        'color': color,
        'diameter_mm': diameter,
        'true_melanoma': true_melanoma.astype(int),
        'predicted_melanoma': predictions.astype(int)
    })

    return df

# Generate biased dataset
df = generate_melanoma_data_with_bias(n_samples=2000)

print(f"Dataset shape: {df.shape}")
print(f"\nSkin tone distribution:")
print(df['skin_tone'].value_counts())
print(f"\nMelanoma prevalence by skin tone:")
print(df.groupby('skin_tone')['true_melanoma'].mean())
print(f"\n(Note: True prevalence is equal across groups - 10%)")

### 1.2 Calculate Stratified Performance Metrics

In [ ]:
def stratified_metrics(df, group_col, y_true_col, y_pred_col):
    """
    Calculate performance metrics stratified by group.

    Parameters:
    -----------
    df : DataFrame
        Data with predictions and group labels
    group_col : str
        Column name for group (e.g., 'skin_tone')
    y_true_col : str
        Column name for true labels
    y_pred_col : str
        Column name for predictions

    Returns:
    --------
    metrics_df : DataFrame
        Stratified metrics by group
    """
    groups = df[group_col].unique()
    results = []

    for group in groups:
        # Filter to this group
        df_group = df[df[group_col] == group]
        y_true = df_group[y_true_col]
        y_pred = df_group[y_pred_col]

        # Calculate confusion matrix
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

        # Calculate metrics
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
        ppv = tp / (tp + fp) if (tp + fp) > 0 else 0
        npv = tn / (tn + fn) if (tn + fn) > 0 else 0
        accuracy = (tp + tn) / (tp + tn + fp + fn)
        f1 = 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) > 0 else 0

        # Positive rate (prediction rate)
        positive_rate = (tp + fp) / len(df_group)

        results.append({
            'Group': group,
            'N': len(df_group),
            'Prevalence': y_true.mean(),
            'Sensitivity': sensitivity,
            'Specificity': specificity,
            'PPV': ppv,
            'NPV': npv,
            'Accuracy': accuracy,
            'F1': f1,
            'Positive_Rate': positive_rate
        })

    return pd.DataFrame(results)

# Calculate stratified metrics
metrics_by_group = stratified_metrics(
    df,
    group_col='skin_tone',
    y_true_col='true_melanoma',
    y_pred_col='predicted_melanoma'
)

print("\nSTRATIFIED PERFORMANCE BY SKIN TONE")
print("=" * 80)
print(metrics_by_group.to_string(index=False))

# Calculate performance gaps
light_sens = metrics_by_group[metrics_by_group['Group'] == 'Light']['Sensitivity'].values[0]
dark_sens = metrics_by_group[metrics_by_group['Group'] == 'Dark']['Sensitivity'].values[0]
sens_gap = light_sens - dark_sens

print(f"\n🚨 FAIRNESS ALERT:")
print(f"  Sensitivity gap: {sens_gap*100:.1f} percentage points")
print(f"  Light skin: {light_sens*100:.1f}% sensitivity")
print(f"  Dark skin: {dark_sens*100:.1f}% sensitivity")
print(f"\n  → Model misses {(1-dark_sens)*100:.0f}% of melanomas on dark skin")
print(f"  → vs only {(1-light_sens)*100:.0f}% on light skin")
print(f"\nClinical Impact (Priya's Story):")
print(f"  Patients with darker skin are MORE LIKELY to have melanoma missed")
print(f"  This perpetuates existing health disparities!")

### 1.3 Visualize Performance Gaps

In [ ]:
# Create comparison plots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

metrics_to_plot = [
    ('Sensitivity', 'Sensitivity (Recall)', 'Are melanomas detected?'),
    ('Specificity', 'Specificity', 'Are benign lesions correctly identified?'),
    ('PPV', 'PPV (Precision)', 'When model says melanoma, is it correct?'),
    ('Accuracy', 'Accuracy', 'Overall correctness')
]

for idx, (metric, ylabel, subtitle) in enumerate(metrics_to_plot):
    ax = axes[idx // 2, idx % 2]

    # Get values
    values = metrics_by_group[metric].values
    groups = metrics_by_group['Group'].values

    # Bar plot
    colors = ['#8B4513', '#F5DEB3']  # Dark brown, light beige
    bars = ax.bar(groups, values, color=colors, edgecolor='black', alpha=0.7)

    # Add value labels
    for bar, val in zip(bars, values):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.02,
                f'{val:.2f}',
                ha='center', va='bottom', fontweight='bold', fontsize=11)

    # Show gap
    if len(values) == 2:
        gap = abs(values[0] - values[1])
        ax.plot([0, 1], [values[0], values[1]], 'r--', linewidth=2, alpha=0.5)
        mid_y = (values[0] + values[1]) / 2
        ax.text(0.5, mid_y, f'Gap: {gap:.2f}',
                ha='center', va='bottom', color='red', fontweight='bold',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

    ax.set_ylabel(ylabel, fontsize=12)
    ax.set_title(f'{metric}\n{subtitle}', fontsize=12, fontweight='bold')
    ax.set_ylim([0, 1.1])
    ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Performance Disparities: Melanoma Detection by Skin Tone',
             fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("  Red dashed line shows performance gap between groups")
print("  Larger gap = more unfair")
print("  Sensitivity gap is most clinically concerning (missed diagnoses)")

---

## Part 2: Formal Fairness Metrics

### Fairness Definitions

**1. Demographic Parity (Statistical Parity)**
- **Definition**: P(Ŷ=1 | A=a) = P(Ŷ=1 | A=b) for all groups a, b
- **Meaning**: Positive prediction rate should be equal across groups
- **Example**: 10% of light skin patients and 10% of dark skin patients are predicted to have melanoma

**2. Equalized Odds**
- **Definition**: P(Ŷ=1 | Y=y, A=a) = P(Ŷ=1 | Y=y, A=b) for all y, a, b
- **Meaning**: TPR (sensitivity) and FPR should be equal across groups
- **Example**: 90% sensitivity and 5% FPR for both light and dark skin

**3. Equality of Opportunity**
- **Definition**: P(Ŷ=1 | Y=1, A=a) = P(Ŷ=1 | Y=1, A=b)
- **Meaning**: TPR (sensitivity) should be equal across groups
- **Example**: Melanomas detected at same rate regardless of skin tone

**Important**: These definitions can conflict! You often can't satisfy all simultaneously.

### 2.1 Calculate Demographic Parity

In [ ]:
def demographic_parity(df, group_col, y_pred_col):
    """
    Calculate demographic parity: positive prediction rates by group.

    Perfect demographic parity: rates are equal across all groups.
    """
    results = {}

    for group in df[group_col].unique():
        df_group = df[df[group_col] == group]
        positive_rate = df_group[y_pred_col].mean()
        results[group] = positive_rate

    # Calculate disparity
    rates = list(results.values())
    max_disparity = max(rates) - min(rates)

    return results, max_disparity

# Calculate demographic parity
dp_rates, dp_disparity = demographic_parity(df, 'skin_tone', 'predicted_melanoma')

print("\nDEMOGRAPHIC PARITY")
print("=" * 60)
print("Positive Prediction Rates (% predicted to have melanoma):")
for group, rate in dp_rates.items():
    print(f"  {group:10s}: {rate*100:.2f}%")

print(f"\nDisparity: {dp_disparity*100:.2f} percentage points")
print(f"\nInterpretation:")
if dp_disparity < 0.05:
    print("  ✅ Good demographic parity (< 5pp difference)")
elif dp_disparity < 0.10:
    print("  ⚠️  Moderate disparity (5-10pp difference)")
else:
    print("  🚨 Large disparity (> 10pp difference)")

print(f"\nNote: Demographic parity doesn't account for actual outcomes!")
print(f"  Even if rates are equal, one group might have higher FP or FN rate.")

### 2.2 Calculate Equalized Odds

In [ ]:
def equalized_odds(df, group_col, y_true_col, y_pred_col):
    """
    Calculate equalized odds: TPR and FPR by group.

    Perfect equalized odds: TPR and FPR are equal across all groups.
    """
    results = {}

    for group in df[group_col].unique():
        df_group = df[df[group_col] == group]
        y_true = df_group[y_true_col]
        y_pred = df_group[y_pred_col]

        # Calculate TPR and FPR
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        tpr = tp / (tp + fn) if (tp + fn) > 0 else 0  # Sensitivity
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0  # 1 - Specificity

        results[group] = {'TPR': tpr, 'FPR': fpr}

    # Calculate disparities
    tprs = [v['TPR'] for v in results.values()]
    fprs = [v['FPR'] for v in results.values()]

    tpr_disparity = max(tprs) - min(tprs)
    fpr_disparity = max(fprs) - min(fprs)

    return results, tpr_disparity, fpr_disparity

# Calculate equalized odds
eo_results, tpr_disparity, fpr_disparity = equalized_odds(
    df, 'skin_tone', 'true_melanoma', 'predicted_melanoma'
)

print("\nEQUALIZED ODDS")
print("=" * 60)
print("True Positive Rate (Sensitivity) and False Positive Rate by Group:")
print(f"{'Group':10s} {'TPR (Sensitivity)':20s} {'FPR (1-Specificity)':20s}")
print("-" * 60)
for group, metrics in eo_results.items():
    print(f"{group:10s} {metrics['TPR']*100:>8.2f}% {metrics['FPR']*100:>20.2f}%")

print(f"\nTPR Disparity: {tpr_disparity*100:.2f} percentage points")
print(f"FPR Disparity: {fpr_disparity*100:.2f} percentage points")

print(f"\nInterpretation:")
if tpr_disparity < 0.05 and fpr_disparity < 0.05:
    print("  ✅ Good equalized odds (< 5pp difference in both TPR and FPR)")
elif tpr_disparity < 0.10 and fpr_disparity < 0.10:
    print("  ⚠️  Moderate disparity (5-10pp difference)")
else:
    print("  🚨 Large disparity (> 10pp difference)")
    print(f"\n  CLINICAL CONCERN: Large TPR gap means melanomas are missed more often")
    print(f"  in one group → delayed diagnosis → worse outcomes")

### 2.3 Calculate Equality of Opportunity

In [ ]:
def equality_of_opportunity(df, group_col, y_true_col, y_pred_col):
    """
    Calculate equality of opportunity: TPR (sensitivity) by group.

    Focuses only on positive class (melanoma cases).
    Perfect equality: all groups have same TPR.
    """
    results = {}

    for group in df[group_col].unique():
        df_group = df[df[group_col] == group]

        # Filter to positive cases only
        df_positive = df_group[df_group[y_true_col] == 1]

        if len(df_positive) > 0:
            tpr = df_positive[y_pred_col].mean()
            results[group] = tpr
        else:
            results[group] = None

    # Calculate disparity
    valid_tprs = [v for v in results.values() if v is not None]
    if len(valid_tprs) > 1:
        disparity = max(valid_tprs) - min(valid_tprs)
    else:
        disparity = 0

    return results, disparity

# Calculate equality of opportunity
eop_results, eop_disparity = equality_of_opportunity(
    df, 'skin_tone', 'true_melanoma', 'predicted_melanoma'
)

print("\nEQUALITY OF OPPORTUNITY")
print("=" * 60)
print("TPR (Sensitivity) by Group:")
for group, tpr in eop_results.items():
    print(f"  {group:10s}: {tpr*100:.2f}%")

print(f"\nDisparity: {eop_disparity*100:.2f} percentage points")

print(f"\nWhy This Matters Most for Healthcare:")
print(f"  Equality of opportunity ensures all patients with disease have equal")
print(f"  OPPORTUNITY to be diagnosed correctly.")
print(f"  {eop_disparity*100:.0f}pp gap means one group has worse survival prospects!")

### 2.4 Visualize All Fairness Metrics

In [ ]:
# Create comprehensive fairness dashboard
fig = plt.figure(figsize=(16, 10))
gs = fig.add_gridspec(3, 2, hspace=0.3, wspace=0.3)

# 1. Demographic Parity
ax1 = fig.add_subplot(gs[0, 0])
groups = list(dp_rates.keys())
values = [dp_rates[g]*100 for g in groups]
colors = ['#8B4513', '#F5DEB3']
bars = ax1.bar(groups, values, color=colors, edgecolor='black', alpha=0.7)
for bar, val in zip(bars, values):
    ax1.text(bar.get_x() + bar.get_width()/2., val + 0.5,
             f'{val:.1f}%', ha='center', va='bottom', fontweight='bold')
ax1.set_ylabel('Positive Prediction Rate (%)', fontsize=11)
ax1.set_title('Demographic Parity\n(Should be equal)', fontsize=12, fontweight='bold')
ax1.set_ylim([0, max(values) * 1.2])
ax1.grid(True, alpha=0.3, axis='y')

# 2. TPR (Sensitivity)
ax2 = fig.add_subplot(gs[0, 1])
tprs = [eo_results[g]['TPR']*100 for g in groups]
bars = ax2.bar(groups, tprs, color=colors, edgecolor='black', alpha=0.7)
for bar, val in zip(bars, tprs):
    ax2.text(bar.get_x() + bar.get_width()/2., val + 1,
             f'{val:.1f}%', ha='center', va='bottom', fontweight='bold')
ax2.set_ylabel('TPR / Sensitivity (%)', fontsize=11)
ax2.set_title('Equality of Opportunity\n(TPR should be equal)', fontsize=12, fontweight='bold')
ax2.set_ylim([0, 100])
ax2.axhline(90, color='green', linestyle='--', alpha=0.5, label='Target: 90%')
ax2.legend()
ax2.grid(True, alpha=0.3, axis='y')

# 3. FPR
ax3 = fig.add_subplot(gs[1, 0])
fprs = [eo_results[g]['FPR']*100 for g in groups]
bars = ax3.bar(groups, fprs, color=colors, edgecolor='black', alpha=0.7)
for bar, val in zip(bars, fprs):
    ax3.text(bar.get_x() + bar.get_width()/2., val + 0.5,
             f'{val:.1f}%', ha='center', va='bottom', fontweight='bold')
ax3.set_ylabel('FPR / 1-Specificity (%)', fontsize=11)
ax3.set_title('Equalized Odds (FPR component)\n(FPR should be equal)', fontsize=12, fontweight='bold')
ax3.set_ylim([0, max(fprs) * 1.5])
ax3.grid(True, alpha=0.3, axis='y')

# 4. Confusion matrices side by side
ax4 = fig.add_subplot(gs[1, 1])
ax4.axis('off')
ax4.text(0.5, 0.9, 'Confusion Matrices by Group',
         ha='center', fontsize=12, fontweight='bold', transform=ax4.transAxes)

for i, group in enumerate(groups):
    df_group = df[df['skin_tone'] == group]
    cm = confusion_matrix(df_group['true_melanoma'], df_group['predicted_melanoma'])

    y_pos = 0.6 - i * 0.3
    ax4.text(0.1, y_pos, f'{group}:', fontweight='bold', transform=ax4.transAxes)

    # Show confusion matrix as text
    text = f"TN={cm[0,0]:4d}  FP={cm[0,1]:4d}\nFN={cm[1,0]:4d}  TP={cm[1,1]:4d}"
    ax4.text(0.3, y_pos-0.05, text, fontsize=10, family='monospace', transform=ax4.transAxes)

# 5. Disparity summary
ax5 = fig.add_subplot(gs[2, :])
ax5.axis('tight')
ax5.axis('off')

summary_data = [
    ['Demographic Parity', f'{dp_disparity*100:.1f}pp',
     'Difference in positive prediction rates',
     '✅ Good' if dp_disparity < 0.05 else '🚨 Gap exists'],
    ['Equality of Opportunity', f'{eop_disparity*100:.1f}pp',
     'Difference in TPR (sensitivity)',
     '✅ Good' if eop_disparity < 0.05 else '🚨 Melanomas missed more in one group'],
    ['Equalized Odds (TPR)', f'{tpr_disparity*100:.1f}pp',
     'Difference in TPR',
     '✅ Good' if tpr_disparity < 0.05 else '🚨 Gap exists'],
    ['Equalized Odds (FPR)', f'{fpr_disparity*100:.1f}pp',
     'Difference in FPR',
     '✅ Good' if fpr_disparity < 0.05 else '⚠️ False alarms differ']
]

table = ax5.table(cellText=summary_data,
                  colLabels=['Fairness Metric', 'Disparity', 'What It Measures', 'Assessment'],
                  cellLoc='left',
                  loc='center',
                  colWidths=[0.25, 0.15, 0.35, 0.25])
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 2)

# Style header
for i in range(4):
    table[(0, i)].set_facecolor('#4CAF50')
    table[(0, i)].set_text_props(weight='bold', color='white')

# Color-code assessment
for i in range(1, 5):
    cell_text = summary_data[i-1][3]
    if '✅' in cell_text:
        table[(i, 3)].set_facecolor('#90EE90')
    elif '🚨' in cell_text:
        table[(i, 3)].set_facecolor('#FFB6C1')
    else:
        table[(i, 3)].set_facecolor('#FFFFE0')

plt.suptitle('Fairness Metrics Dashboard: Melanoma Detection AI',
             fontsize=16, fontweight='bold', y=0.98)
plt.show()

print("\nDashboard Interpretation:")
print("  Top row: Shows if prediction rates (DP) and detection rates (TPR) are equal")
print("  Middle row: Shows false positive rates and confusion matrices")
print("  Bottom table: Summary of all fairness gaps")

---

## Part 3: Understanding Trade-offs

### The Impossibility Theorem

**Key Insight**: You often **cannot** satisfy all fairness criteria simultaneously!

**Example:**
- If disease prevalence differs between groups
- You cannot have both demographic parity AND equalized odds
- Must choose which fairness definition matters most for your application

**Healthcare Priority**: Usually **Equality of Opportunity** (equal TPR)
- All patients with disease deserve equal chance of diagnosis
- Missing melanoma in one group is unacceptable
- False positives, while costly, are less harmful than false negatives

### 3.1 Fairness vs Accuracy Trade-off

In [ ]:
print("\nFAIRNESS VS ACCURACY TRADE-OFF")
print("=" * 60)

# Calculate overall metrics
overall_accuracy = accuracy_score(df['true_melanoma'], df['predicted_melanoma'])
overall_sensitivity = metrics_by_group['Sensitivity'].mean()
overall_specificity = metrics_by_group['Specificity'].mean()

print(f"Current Model (biased):")
print(f"  Overall Accuracy: {overall_accuracy:.3f}")
print(f"  Average Sensitivity: {overall_sensitivity:.3f}")
print(f"  Sensitivity Gap: {eop_disparity:.3f}")

print(f"\nScenario 1: Optimize for overall accuracy (status quo)")
print(f"  → High accuracy, but unfair to dark skin patients")
print(f"  → Some groups have much worse outcomes")

print(f"\nScenario 2: Enforce equal sensitivity across groups")
print(f"  → Might reduce overall accuracy slightly")
print(f"  → But ensures all patients have equal chance of diagnosis")
print(f"  → More ethical and legally defensible")

print(f"\nScenario 3: Lower threshold for dark skin patients")
print(f"  → Increases sensitivity for dark skin (fewer false negatives)")
print(f"  → Increases false positives for dark skin")
print(f"  → Trade-off: More biopsies, but fewer missed melanomas")

print(f"\n💡 Key Decision:")
print(f"  In healthcare, we often accept LOWER overall accuracy")
print(f"  to achieve FAIRNESS across groups.")
print(f"  Why? Because systematic bias can worsen health disparities!")

### 3.2 Mitigation Strategies

In [ ]:
print("\nSTRATEGIES TO REDUCE BIAS")
print("=" * 60)

strategies = [
    {
        'Strategy': 'Collect more diverse training data',
        'Description': 'Include equal representation of all skin tones',
        'When': 'Before training (pre-processing)',
        'Effectiveness': 'High',
        'Priya Example': 'Train on 50% dark skin images instead of 10%'
    },
    {
        'Strategy': 'Data augmentation for underrepresented groups',
        'Description': 'Oversample minority group or generate synthetic examples',
        'When': 'Before training (pre-processing)',
        'Effectiveness': 'Medium-High',
        'Priya Example': '5x augmentation of dark skin images'
    },
    {
        'Strategy': 'Fairness constraints during training',
        'Description': 'Add penalty for unequal TPR across groups',
        'When': 'During training (in-processing)',
        'Effectiveness': 'High',
        'Priya Example': 'Loss function penalizes sensitivity gap'
    },
    {
        'Strategy': 'Group-specific thresholds',
        'Description': 'Use different decision thresholds per group',
        'When': 'After training (post-processing)',
        'Effectiveness': 'Medium',
        'Priya Example': 'Threshold=0.3 for dark skin, 0.5 for light skin'
    },
    {
        'Strategy': 'Train separate models per group',
        'Description': 'Specialized model for each group',
        'When': 'Architecture choice',
        'Effectiveness': 'High (but data-hungry)',
        'Priya Example': 'One model for light skin, one for dark skin'
    },
    {
        'Strategy': 'Regular fairness audits',
        'Description': 'Monitor stratified performance in production',
        'When': 'Deployment & monitoring',
        'Effectiveness': 'Critical',
        'Priya Example': 'Monthly reports on sensitivity by skin tone'
    }
]

df_strategies = pd.DataFrame(strategies)
print(df_strategies[['Strategy', 'When', 'Effectiveness']].to_string(index=False))

print("\n\n📋 Recommended Approach (Multi-pronged):")
print("  1. Collect more diverse data (most important!)")
print("  2. Add fairness constraints during training")
print("  3. Use group-specific thresholds if needed")
print("  4. Monitor fairness metrics continuously in production")
print("  5. Get feedback from diverse patient populations")

---

## Key Takeaways

### 1. Always Stratify by Protected Attributes

**Before deployment:**
- Calculate performance metrics separately for each demographic group
- Look for gaps in sensitivity, specificity, PPV
- Sensitivity gaps are most clinically concerning (missed diagnoses)

**Protected attributes to check:**
- Race/ethnicity
- Gender
- Age groups
- Socioeconomic status
- Geographic location

### 2. Fairness Definitions

**Demographic Parity**
- Equal positive prediction rates across groups
- P(Ŷ=1 | A=a) = P(Ŷ=1 | A=b)
- Doesn't account for actual outcomes

**Equalized Odds**
- Equal TPR and FPR across groups
- P(Ŷ=1 | Y=y, A=a) = P(Ŷ=1 | Y=y, A=b)
- Most comprehensive fairness measure

**Equality of Opportunity**
- Equal TPR (sensitivity) across groups
- P(Ŷ=1 | Y=1, A=a) = P(Ŷ=1 | Y=1, A=b)
- **Most important for healthcare** (equal diagnosis rates)

### 3. Healthcare Priority: Equal Sensitivity

**Why equality of opportunity matters most:**
- Missing disease in one group → worse health outcomes
- Perpetuates existing health disparities
- Violates principle of equal care
- Legal and ethical liability

**Acceptable trade-off:**
- May accept slightly higher FP rate in one group
- To achieve equal sensitivity across groups
- Better to biopsy more than miss melanomas

### 4. Common Sources of Bias

**Training data:**
- Underrepresentation of minority groups
- Historical disparities in healthcare access
- Different disease presentation in different groups

**Feature selection:**
- Proxies for protected attributes (zip code → race)
- Missing features that vary by group

**Evaluation:**
- Only reporting overall metrics
- Not stratifying by protected attributes

### 5. Mitigation Strategies

**Pre-processing (best):**
- Collect diverse, representative training data
- Oversample underrepresented groups
- Ensure equal data quality across groups

**In-processing:**
- Add fairness constraints to loss function
- Train with fairness-aware algorithms
- Adversarial debiasing

**Post-processing:**
- Group-specific decision thresholds
- Calibration per group
- Separate models per group

**Monitoring:**
- Regular fairness audits in production
- Track stratified metrics over time
- Get feedback from affected communities

### 6. Regulatory and Ethical Considerations

**FDA guidance:**
- Must demonstrate performance across demographic subgroups
- Document mitigation strategies for bias

**Legal:**
- Discriminatory algorithms violate civil rights laws
- Liability for disparate impact

**Ethical:**
- Principle of justice (fair distribution of benefits/harms)
- Do no harm (including systematic harm to groups)
- Trust and acceptance

---

## Exercises

### Exercise 1: Calculate Fairness Metrics
Given a sepsis prediction model:
- Overall sensitivity: 85%
- White patients: 90% sensitivity
- Black patients: 75% sensitivity

1. Calculate the equality of opportunity disparity
2. If prevalence is 15% in both groups, calculate demographic parity
3. Which fairness criterion is violated?
4. What is the clinical impact of this bias?

### Exercise 2: Evaluate Mitigation Strategies
You have a melanoma detection model with:
- Light skin: 92% sensitivity, 88% specificity
- Dark skin: 70% sensitivity, 88% specificity

Propose three mitigation strategies:
1. What would you do before retraining?
2. What loss function modification could help?
3. What post-processing could you apply?
4. How would you evaluate if your fixes worked?

### Exercise 3: Trade-off Analysis
You can improve dark skin sensitivity from 70% to 85% by:
- Option A: Lower threshold → increases FP by 10pp
- Option B: Retrain with balanced data → overall accuracy drops 2%

Questions:
1. Which option would you choose and why?
2. How would you explain this to stakeholders?
3. What additional data would help you decide?
4. Are there regulatory considerations?

### Exercise 4: Real-World Case Study
Implement stratified evaluation on a real dataset:
1. Load a healthcare dataset with demographic information
2. Train a classification model
3. Calculate all fairness metrics by group
4. Create a fairness dashboard
5. Write a 1-page fairness assessment report

### Exercise 5: Impossibility Theorem
Create a scenario where:
1. Disease prevalence differs between groups (5% vs 15%)
2. You achieve perfect equalized odds
3. Show that demographic parity is violated
4. Explain why this is mathematically impossible to fix
5. Discuss which fairness criterion should take priority

---

**This completes Chapter 5: Model Evaluation!**

**Next Chapter**: 6. Medical Imaging with Deep Learning